# Segmentación semántica de SuperTuxKart

Prueba el modelo entrenado sobre tus propias imágenes. **No hace falta instalar nada**: ejecuta las celdas en orden con `Shift+Enter`, o usa *Entorno de ejecución → Ejecutar todas*.

El modelo asigna a cada píxel una de 7 clases: `background`, `track`, `kart`, `pickup`, `nitro`, `bomb`, `projectile`.


## 1. Preparar (unos 30 segundos)


In [ ]:
#@title Descargar el código y el modelo
REPO  = 'https://github.com/ferjozsot23/vision-supertuxkart'  #@param {type:"string"}
MODEL = 'https://github.com/ferjozsot23/vision-supertuxkart/releases/latest/download/model.th'  #@param {type:"string"}
# MODEL: URL del .th (pestaña Releases del repositorio).
# Si se deja vacío, se busca el modelo dentro del propio repositorio.

import os, sys, subprocess
if not os.path.isdir('stk'):
    subprocess.run(['git','clone','--depth','1',REPO,'stk'], check=True)
os.chdir('/content/stk') if os.path.isdir('/content/stk') else os.chdir('stk')
sys.path.insert(0, os.getcwd())

if MODEL:
    os.environ['STK_MODEL_URL'] = MODEL
    from predict import ensure_model
    ensure_model('model.th')

import torch
assert os.path.exists('model.th'), (
    'Falta model.th. Pega en MODEL la URL del archivo desde la pestaña Releases.')
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())


In [ ]:
#@title Cargar el modelo
import torch, numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from models import load_model
from utils import CLASS_NAMES, PALETTE, label_to_color, overlay

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = load_model('model.th', device=device)
n = sum(p.numel() for p in model.parameters())
print('U-Net cargada en %s — %.1f M parámetros' % (device, n/1e6))
print('La imagen se pasa CRUDA en [0,1]; la normalización va dentro del modelo.')
print('Acepta cualquier resolución de entrada.')


In [ ]:
#@title Función de segmentación
def segmentar(ruta, mostrar=True):
    """Devuelve (H,W) uint8 con el índice de clase de cada píxel."""
    img = Image.open(ruta).convert('RGB')
    x = torch.from_numpy(np.asarray(img, np.uint8).copy())\
             .permute(2,0,1).float().div_(255.)[None].to(device)
    with torch.no_grad():
        pred = model(x).argmax(1)[0].cpu().numpy().astype(np.uint8)

    if mostrar:
        rgb = np.asarray(img)
        fig, ax = plt.subplots(1, 3, figsize=(15, 5.4))
        for a, d, t in zip(ax, [rgb, label_to_color(pred),
                                overlay(rgb.transpose(2,0,1)/255., pred)],
                           ['entrada', 'segmentación', 'superposición']):
            a.imshow(d); a.set_title(t, fontsize=12); a.axis('off')
        presentes = sorted(int(c) for c in np.unique(pred))
        fig.legend([plt.Rectangle((0,0),1,1, fc=PALETTE[c]/255.) for c in presentes],
                   [CLASS_NAMES[c] for c in presentes],
                   loc='lower center', ncol=7, frameon=False, fontsize=11)
        fig.suptitle(os.path.basename(ruta), fontsize=13)
        fig.tight_layout(rect=[0,.06,1,.97]); plt.show()
    return pred

print('listo')


## 2. Probar con las imágenes de ejemplo

Vienen en el repositorio. Las tres primeras son de circuitos que el modelo **nunca vio durante el entrenamiento**.


In [ ]:
for f in sorted(os.listdir('ejemplos')):
    segmentar(os.path.join('ejemplos', f))


## 3. Probar con tus propias imágenes

Ejecuta la celda y selecciona uno o varios archivos de tu ordenador. Vale cualquier resolución.


In [ ]:
from google.colab import files
subidas = files.upload()
for nombre in subidas:
    segmentar(nombre)


## 4. Obtener las máscaras como archivo

Guarda la máscara cruda (1 canal, valores 0–6) de cada imagen subida y la descarga en un zip.


In [ ]:
os.makedirs('salida', exist_ok=True)
for nombre in subidas:
    pred = segmentar(nombre, mostrar=False)
    Image.fromarray(pred).save('salida/%s_mask.png' % os.path.splitext(nombre)[0])
    print('salida/%s_mask.png' % os.path.splitext(nombre)[0])

!zip -qr mascaras.zip salida && echo 'mascaras.zip creado'
files.download('mascaras.zip')


---

### Uso por línea de comandos

```bash
pip install -r requirements.txt
python predict.py mis_imagenes/ --out resultados/ --masks
```

### En tu propio código

```python
from models import load_model
model = load_model('model.th')       # sin argumentos
pred  = model(x).argmax(1)           # x: (B,3,H,W) float en [0,1]
```
